# Week 5: IDS and one-page Nôm recognition preparation

This notebook uses the pinned course dictionaries. It demonstrates **text-level** IDS tokenization; it does not segment radical regions from pixels. Run the resource download command in the handout before starting.

## 1. IDS as a prefix expression

An IDC comes before its operands: `⿰日月` means 日 is left of 月. Binary IDCs such as `⿰`, `⿱`, `⿴`, and `⿻` take two operands; `⿲` and `⿳` take three. A direct CJKVI IDS describes one layer. The recognizer's expanded IDS recursively replaces components until it reaches its terminal vocabulary.

In [ ]:
import json
import os
from urllib.parse import quote
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

import nom_ocr_lab
from nom_ocr_lab.annotations import read_line_annotations
from nom_ocr_lab.recognition import (
    load_expanded_ids,
    load_vocabulary,
    reverse_ids,
    text_to_ids,
)

workspace_value = os.environ.get("LAB_WORKSPACE")
if not workspace_value:
    raise RuntimeError("Set LAB_WORKSPACE before starting Jupyter")
workspace = Path(workspace_value)
course_root = Path(nom_ocr_lab.__file__).resolve().parents[2]
ids_root = course_root / "nom-ids"
if not ids_root.is_dir():
    raise RuntimeError(f"Bundled IDS resources are missing: {ids_root}")
sample_root = workspace / "datasets" / "nomnaocr-sample"
print(f"Lab workspace: {workspace}")
print(f"Bundled IDS resources: {ids_root}")

In [ ]:
expanded = load_expanded_ids(ids_root / "ids_exp.txt")
vocabulary = load_vocabulary(ids_root / "vocab_ids.txt")
ids_to_characters = reverse_ids(expanded)
print(f"Expanded character entries: {len(expanded):,}")
print(f"Recognizer tokens, in checkpoint order: {len(vocabulary):,}")

## 2. Runnable reference converter and exact Chinese check

`text_to_ids` emits one structured record per non-whitespace Unicode code point. Copying the whole string into one field would lose the character boundaries needed later. Reuse this provided converter in your team repository; each record also includes a direct `zi_tools_url`.

In [ ]:
expected = {
    "休": "⿰亻木",
    "好": "⿰女子",
    "字": "⿱宀子",
    "明": "⿰日月",
    "林": "⿰木木",
}
reference = text_to_ids("休好字明林", expanded)
observed = {record.character: record.ids for record in reference}
assert observed == expected, (observed, expected)
for record in reference:
    candidates = ids_to_characters[record.ids]
    zi_tools_url = f"https://zi.tools/zi/{quote(record.character)}"
    print(record.codepoint, record.character, record.ids, list(record.tokens), candidates, zi_tools_url)
print("Five Chinese reference conversions passed.")

Before continuing, write `明`, `休`, and one nested character as IDS by hand. Compare the spatial relation you chose with the direct column printed above, then explain why a direct and an expanded IDS can differ.

## 3. Audit the durable Week 1 Nôm corpus, then choose one page

The next cell audits every durable NomNaOCR page for characters missing from `ids_exp.txt`. Set `WEEK05_PAGE` to a path relative to `nomnaocr-sample/images` if you want a page other than the first sorted image for the later crop exercise. The same relative path locates its line-label file.

In [ ]:
image_root = sample_root / "images"
image_suffixes = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
available = sorted(
    path.relative_to(image_root)
    for path in image_root.rglob("*")
    if path.is_file() and path.suffix.lower() in image_suffixes
)
if not available:
    raise RuntimeError(f"No Week 1 sample images found below {image_root}")
requested = os.environ.get("WEEK05_PAGE")
relative_image = Path(requested) if requested else available[0]
if relative_image not in available:
    raise RuntimeError(f"WEEK05_PAGE is not in the durable sample: {relative_image}")
image_path = image_root / relative_image
line_path = sample_root / "line_labels" / relative_image.with_suffix(".txt")
annotations = read_line_annotations(line_path)
print(f"Selected page: {relative_image}")
print(f"Line transcriptions: {len(annotations)}")

In [ ]:
corpus_records = []
oov_pages = Counter()
for relative_path in available:
    corpus_line_path = sample_root / "line_labels" / relative_path.with_suffix(".txt")
    if not corpus_line_path.is_file():
        raise RuntimeError(f"Missing line labels for durable sample page: {relative_path}")
    page_records = [
        record
        for annotation in read_line_annotations(corpus_line_path)
        for record in text_to_ids(annotation.text, expanded)
    ]
    corpus_records.extend(page_records)
    for character in {record.character for record in page_records if record.status == "oov"}:
        oov_pages[character] += 1

covered_total = sum(record.status == "covered" for record in corpus_records)
character_total = len(corpus_records)
coverage = covered_total / character_total if character_total else 0.0
oov_counts = Counter(record.character for record in corpus_records if record.status == "oov")
print(f"Corpus pages scanned: {len(available)}")
print(f"Corpus IDS coverage: {covered_total}/{character_total} = {coverage:.2%}")
print(
    f"Missing IDS: {sum(oov_counts.values())} occurrences of "
    f"{len(oov_counts)} distinct characters across {sum(oov_pages.values())} page-character pairs"
)
print("top | char | Unicode | occurrences | pages | zi.tools")
for rank, (character, occurrences) in enumerate(oov_counts.most_common(5), 1):
    print(
        f"{rank:>3} | {character} | U+{ord(character):04X} | {occurrences:>11} | "
        f"{oov_pages[character]:>5} | https://zi.tools/zi/{quote(character)}"
    )
if not oov_counts:
    print("No missing IDS characters in this durable corpus.")

## Submission reflection

1. What information does `⿰` add that an unordered radical set would lose?
2. Why must `vocab_ids.txt` retain its original order?
3. Which page characters were OOV, and were they uncertain marks, punctuation, or ideographs?
4. What direct/expanded or glyph-region differences appeared in zi.tools?
5. Why would pairing the first detected box with the first transcription character create false ground truth?